In [0]:
pip install void

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
pip install void

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import void
import uuid

In [0]:
spark.sql("USE CATALOG novacart_adb")
spark.sql("create database if not exists gold_schema")
gold_run_id=str(uuid.uuid4())
print(gold_run_id)
run_ts_str=datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(run_ts_str)
run_date_str=datetime.now().strftime("%Y-%m-%d")
print(run_date_str)
print("current Gold Run ID:",gold_run_id)
print("Run Timestamp Folder:",run_ts_str)
print("Run Date Folder:",run_date_str)

a2d28d0d-24b9-4ce9-a46b-dd90a28447ee
2026-04-06 18:45:33
2026-04-06
current Gold Run ID: a2d28d0d-24b9-4ce9-a46b-dd90a28447ee
Run Timestamp Folder: 2026-04-06 18:45:33
Run Date Folder: 2026-04-06


2.gold control table

In [0]:
spark.sql("""
          create table if not exists novacart_adb.gold_schema.processing_control
          (
              layer string,
              entity_name string,
              last_processes_silver_run_id string,
              last_processed_silver_run_ts timestamp,
              rows_merged bigint,
              run_status string,
              gold_run_id string,
              updated_at timestamp
          )
          using delta
            """)

DataFrame[]

3.Helper function

In [0]:
def upsert_to_gold(df_source,target_table,join_key):
    if spark.catalog.tableExists(target_table):
        dt=DeltaTable.forName(spark,target_table)
        (dt.alias("target")
        .merge(df_source.alias("source"),f"target.{join_key}=source.{join_key}")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    else:
        df_source.write.format("delta").saveAsTable(target_table)

In [0]:
def get_last_processed_silver_ts(entity_name:str):
    ctrl=(
        spark.table("novacart_adb.gold_schema.processing_control")
        .filter((F.col("layer")=="gold")&
                (F.col("entity_name")==entity_name)&
                (F.col("run_status")=="SUCCESS"))
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )
    rows=ctrl.collect()
    if len(rows)==0:
        return None
    return rows[0]["last_processed_silver_run_ts"]

In [0]:
def upsert_gold_control(entity_name,last_processed_silver_run_id,last_processed_silver_run_ts,rows_merged):
    ctrl_df=spark.createDataFrame(
        [("gold",entity_name,last_processed_silver_run_id,last_processed_silver_run_ts,int(rows_merged),"SUCCESS",gold_run_id,datetime.now())],
        schema="""
        layer string,
        entity_name string,
        last_processes_silver_run_id string,
        last_processed_silver_run_ts timestamp,
        rows_merged bigint,
        run_status string,
        gold_run_id string,
        updated_at timestamp
        """)
    dt=DeltaTable.forName(spark,"novacart_adb.gold_schema.processing_control")
    (dt.alias("target")
     .merge(ctrl_df.alias("source"),f"target.entity_name=source.entity_name")
     .whenMatchedUpdate(set={
         "last_processes_silver_run_id":"source.last_processes_silver_run_id",
         "last_processed_silver_run_ts":"source.last_processed_silver_run_ts",
         "rows_merged":"source.rows_merged",
         "run_status":"source.run_status",
         "gold_run_id":"source.gold_run_id",
         "updated_at":"source.updated_at"
         })
     .whenNotMatchedInsertAll()
     .execute())

In [0]:
last_gold_ts=get_last_processed_silver_ts("orders_information")
print("Last Processed Silver Timestamp for Gold:",last_gold_ts)
silver_orders_current=spark.read.table("novacart_adb.silver_schema.orders_transformed")
silver_products_current=spark.read.table("novacart_adb.silver_schema.products_transformed")
silver_payments_current=spark.read.table("novacart_adb.silver_schema.payments_transformed")
if last_gold_ts is None:
    changed_orders=silver_orders_current
    changed_products=silver_products_current
    changed_payments=silver_payments_current
else:
    changed_orders=silver_orders_current.filter(F.col("updated_at")>F.lit(last_gold_ts))
    changed_products=silver_products_current.filter(F.col("updated_at")>F.lit(last_gold_ts))
    changed_payments=silver_payments_current.filter(F.col("updated_at")>F.lit(last_gold_ts))

changed_orders_count=changed_orders.count()
changed_products_count=changed_products.count()
changed_payments_count=changed_payments.count()
print(f"Number of changed orders={changed_orders_count}")
print(f"Number of changed products={changed_products_count}")
print(f"Number of changed payments={changed_payments_count}")

Last Processed Silver Timestamp for Gold: None
Number of changed orders=0
Number of changed products=111
Number of changed payments=391


In [0]:
impacted_from_orders= changed_orders.select("order_id").distinct()
impacted_from_payments=changed_payments.select("order_id").distinct()
orders_products=(
    changed_products.alias("products")
    .join(silver_orders_current.alias("orders"),F.col("products.product_id")==F.col("orders.product_id"),"inner")
.select(F.col("orders.order_id")).distinct()
)
impacted_order_ids=(
    impacted_from_orders.union(impacted_from_payments.union(orders_products).distinct())
)
print("impacted_order_ids=",impacted_order_ids.count())
display(impacted_order_ids.orderBy(F.col("order_id")))

impacted_order_ids= 391


order_id
200001
200002
200003
200004
200005
200006
200007
200008
200009
200010


6.Bulid gold delta for impacted orders

In [0]:
impacted_orders=(
    silver_orders_current.alias("orders")
    .join(impacted_order_ids.alias("impacted"),"order_id","inner")
)
gold_delta=(
    impacted_orders.alias("orders")
    .join(silver_products_current.alias("products"),F.col("orders.product_id")==F.col("products.product_id"),"inner")
    .join(silver_payments_current.alias("payments"),F.col("orders.order_id")==F.col("payments.order_id"),"inner")
    .select(F.col("orders.order_id"),
            F.col("orders.customer_id"),
            F.col("products.product_id"),
            F.col("products.product_name"),
            F.col("products.category"),
            F.col("products.price").alias("products_price"),
            F.col("orders.order_status"),
            F.col("orders.order_amount"),
            F.col("payments.payment_id"),
            F.col("payments.payment_status"),
            F.col("payments.paid_amount"),
            F.col("orders.order_date"),
            F.col("orders.order_month"),
            F.col("orders.order_year"),
            F.greatest(F.col("orders.updated_at").cast("timestamp"),
                       F.col("products.updated_at").cast("timestamp"),
                       F.col("payments.processed_at").cast("timestamp")
                       ).alias("gold_update_ts")
           )

.dropDuplicates(["order_id"])
.withColumn(
    "payment_completion_ratio",
    F.when(
        F.col("order_amount")>0,
        F.col("paid_amount")/F.col("order_amount")
        ).otherwise(F.lit(0.0))
    )
.withColumn(
    "payment_state",
    F.when(F.col("order_amount")==0,"Invaild_order_amount")
    .when(F.col("payment_completion_ratio")==0,"Unpaid")
    .when(F.col("payment_completion_ratio")==1,"Paid")
    .when(F.col("payment_completion_ratio")<1,"Partially_paid")
    .when(F.col("payment_completion_ratio")>1,"Overpaid")
    )
    .withColumn("gold_updated_date",F.col("gold_update_ts"))
    .withColumn("gold_run_id",F.lit(gold_run_id))
)
print("gold_delta_rows=",gold_delta.count())
display(gold_delta)

gold_delta_rows= 0


order_id,customer_id,product_id,product_name,category,products_price,order_status,order_amount,payment_id,payment_status,paid_amount,order_date,order_month,order_year,gold_update_ts,payment_completion_ratio,payment_state,gold_updated_date,gold_run_id


7.Merge gold current-state table

In [0]:
if gold_delta.count()>0:
    upsert_to_gold(gold_delta,"novacart_adb.gold_schema.orders_information","order_id")
    upsert_gold_control("orders_information",gold_run_id,last_gold_ts,gold_delta.count())
else:
    print("No new rows to insert into gold table")

In [0]:
%sql
    
select * from novacart_adb.gold_schema.orders_information;

order_id,customer_id,product_id,product_name,category,products_price,order_status,order_amount,payment_id,payment_status,paid_amount,order_date,order_month,order_year,gold_update_ts,payment_completion_ratio,payment_state,gold_updated_date,gold_run_id
200004,5004,1004,PRODUCT 4,ELECTRONICS,14.0,PLACED,54.0,900004,SUCCESS,100.0,2026-02-01,2,2026,2026-02-01T11:04:00.000Z,1.8518518518518519,Overpaid,2026-02-01T11:04:00.000Z,a2d28d0d-24b9-4ce9-a46b-dd90a28447ee
200051,5051,1051,PRODUCT 51,ELECTRONICS,61.0,PLACED,101.0,900051,SUCCESS,100.0,2026-02-01,2,2026,2026-02-01T11:51:00.000Z,0.9900990099009901,Partially_paid,2026-02-01T11:51:00.000Z,a2d28d0d-24b9-4ce9-a46b-dd90a28447ee
200108,5108,1108,PROD_108,ELECTRNICS,28.0,CANCELLED,5800.0,900108,FAILED,100.0,2026-02-01,2,2026,2026-02-01T12:48:00.000Z,0.017241379310344827,Partially_paid,2026-02-01T12:48:00.000Z,a2d28d0d-24b9-4ce9-a46b-dd90a28447ee
200111,5111,1111,PRODUCT 111,ELECTRONICS,31.0,PLACED,61.0,900111,SUCCESS,100.0,2026-02-01,2,2026,2026-02-01T12:51:00.000Z,1.639344262295082,Overpaid,2026-02-01T12:51:00.000Z,a2d28d0d-24b9-4ce9-a46b-dd90a28447ee
200166,5166,1046,PRODUCT 46,ELECTRONICS,56.0,PLACED,116.0,900166,SUCCESS,100.0,2026-02-01,2,2026,2026-02-01T13:46:00.000Z,0.8620689655172413,Partially_paid,2026-02-01T13:46:00.000Z,a2d28d0d-24b9-4ce9-a46b-dd90a28447ee
200169,5169,1049,PRODUCT 49,ELECTRONICS,59.0,PLACED,119.0,900169,SUCCESS,100.0,2026-02-01,2,2026,2026-02-01T13:49:00.000Z,0.8403361344537815,Partially_paid,2026-02-01T13:49:00.000Z,a2d28d0d-24b9-4ce9-a46b-dd90a28447ee
200205,5005,1085,PRODUCT 85,ELECTRONICS,95.0,PLACED,55.0,900205,SUCCESS,100.0,2026-02-01,2,2026,2026-02-01T14:25:00.000Z,1.8181818181818181,Overpaid,2026-02-01T14:25:00.000Z,a2d28d0d-24b9-4ce9-a46b-dd90a28447ee
200259,5059,1019,PRODUCT 19,ELECTRONICS,29.0,PLACED,109.0,900259,SUCCESS,100.0,2026-02-01,2,2026,2026-02-01T15:19:00.000Z,0.9174311926605505,Partially_paid,2026-02-01T15:19:00.000Z,a2d28d0d-24b9-4ce9-a46b-dd90a28447ee
200287,5087,1047,PRODUCT 47,ELECTRONICS,57.0,PLACED,137.0,900287,SUCCESS,100.0,2026-02-01,2,2026,2026-02-01T15:47:00.000Z,0.7299270072992701,Partially_paid,2026-02-01T15:47:00.000Z,a2d28d0d-24b9-4ce9-a46b-dd90a28447ee
200323,5123,1083,PRODUCT 83,ELECTRONICS,93.0,PLACED,73.0,900323,SUCCESS,100.0,2026-02-01,2,2026,2026-02-01T16:23:00.000Z,1.36986301369863,Overpaid,2026-02-01T16:23:00.000Z,a2d28d0d-24b9-4ce9-a46b-dd90a28447ee


8.maintain gold scd type 2 history

In [0]:
if not spark.catalog.tableExists("novacart_adb.gold_schema.orders_information_scd2"):
    spark.sql("""
                create table novacart_adb.gold_schema.orders_information_scd2
                using delta
                as
                select *,
                cast(null as timestamp) as valid_from_ts,
                cast(null as timestamp) as valid_to_ts,
                true as is_current
                from novacart_adb.gold_schema.orders_information where 1=0""")
if gold_delta.count()>0:
    gold_delta.createOrReplaceTempView("gold_delta_veiw")
    spark.sql("""
                merge into novacart_adb.gold_schema.orders_information_scd2 as target
                using gold_delta_veiw as source
                on target.order_id=source.order_id and target.is_current=true
                when matched and (
                    not(target.order_status<=>source.order_status)
                    or not(target.order_amount<=>source.order_amount)
                    or not(target.paid_amount<=>source.paid_amount)
                    or not(target.payment_id<=>source.payment_id)
                    or not(target.category<=>source.category)
                    or not(target.product_name<=>source.product_name)
                    or not(target.products_price<=>source.products_price)
                ) then update set
                    is_current = false,
                    valid_to_ts = source.gold_update_ts
                """)
    spark.sql("""
                insert into novacart_adb.gold_schema.orders_information_scd2
                select source.*,
                  source.gold_update_ts as valid_from_ts,
                  cast(null as timestamp) as valid_to_ts,
                  true as is_current
                from gold_delta_veiw as source
                left outer join novacart_adb.gold_schema.orders_information_scd2 as target
                on source.order_id=target.order_id and target.is_current=true
                where target.order_id is null or (
                    not(target.order_id<=>source.order_id)
                    or not(target.order_status<=>source.order_status)
                    or not(target.order_amount<=>source.order_amount)
                    or not(target.paid_amount<=>source.paid_amount)
                    or not(target.payment_id <=> source.payment_id)
                    or not(target.category <=> source.category)
                    or not(target.product_name <=> source.product_name))
                """)